# **영화 리뷰 감성 판별 실습**
- 영화 리뷰가 긍정인지, 부정인지 학습
- 레이블 : 긍정(1), 부정(0)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import re

1. IMDB data load

In [2]:
imdb = keras.datasets.imdb

# 자주 등장하는 상위 10,000개의 단어만 선택적으로 로드
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

print("훈련 데이터 길이:", len(x_train), "테스트 데이터 길이:", len(x_test))
print("첫 번째 데이터 레이블 (y_train[0]):", y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
훈련 데이터 길이: 25000 테스트 데이터 길이: 25000
첫 번째 데이터 레이블 (y_train[0]): 1


2. 영화 리뷰 텍스트 복원
- Full Text 출력
- imdb.get_word_index() : 각 단어와 인덱스에 저장된 딕셔너리 반환
- 딕셔너리의 키와 값의 순서를 바꿈

In [3]:
# 단어 -> 정수 인덱스 딕셔너리
word_to_index = imdb.get_word_index()

# 특수 용도 토큰 배정을 위해 기존 단어 인덱스를 3씩 뒤로 이동
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}
word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

# 키와 값의 순서를 바꾸어 인덱스로부터 단어를 찾을 수 있게 변환
index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

# 첫 번째 리뷰 복원 결과 출력
decoded_review = ' '.join([index_to_word.get(index, '?') for index in x_train[0]])
print("\n--- 복원된 첫 번째 리뷰 원문 ---")
print(decoded_review[:300] + "...\n")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- 복원된 첫 번째 리뷰 원문 ---
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the...



3. 전처리 (Padding)

In [4]:
# 리뷰 문장의 크기가 각자 다르므로 최대 길이를 100 단어로 일정하게 제한
x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

print("패딩 적용 후 x_train 셰이프:", x_train.shape)

패딩 적용 후 x_train 셰이프: (25000, 100)


4. Sequential Model Building + Embedding
- Embedding : 정수 단어 배열을 입력받아 각 단어를 64차원의 밀집 정밀 벡터 공간으로 매핑

In [5]:
vocab_size = 10000

model = keras.Sequential([
    keras.layers.Embedding(vocab_size, 64, input_length=100),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),                              # Dropout : 과적합 방지
    keras.layers.Dense(1, activation='sigmoid')             # sigmoid : 이진 분류
])

# 컴파일 : 손실 함수와 최적화 도구 지정
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

5. Model fit + Test

In [6]:
# 모델 학습
history = model.fit(x_train, y_train,
                    batch_size=64,
                    epochs=20,
                    verbose=1,
                    validation_data=(x_test, y_test))

# 최종 검증 성능 측정
results = model.evaluate(x_test, y_test, verbose=2)
print(f"\n[테스트 데이터 평가] Loss: {results[0]:.4f}, Accuracy: {results[1]:.4f}\n")

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - accuracy: 0.7618 - loss: 0.4662 - val_accuracy: 0.8463 - val_loss: 0.3433
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9347 - loss: 0.1794 - val_accuracy: 0.8310 - val_loss: 0.4064
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9924 - loss: 0.0319 - val_accuracy: 0.8310 - val_loss: 0.5321
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9993 - loss: 0.0059 - val_accuracy: 0.8328 - val_loss: 0.6178
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.9998 - loss: 0.0022 - val_accuracy: 0.8334 - val_loss: 0.6701
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 1.0000 - loss: 8.2877e-04 - val_accuracy: 0.8321 - val_loss: 0.7281
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9991 - loss: 0.0035 - val_accuracy: 0.8304 - val_loss: 0.7722
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9998 - loss: 9.2831e

6. 새로운 임의의 리뷰 감성 판별 (추론)

In [7]:
# 테스트 문장 설정 ("Back to the Future (1985)" 실제 영화 평)
review = ("What can I say about this movie that was already said? It is my "
          "favorite time travel sci-fi, adventure epic comedy in the 80's and I "
          "love this movie to death! When I saw this movie I was thrown out by its theme. "
          "An excellent sci-fi, adventure epic, I LOVE the 80s. It's simple the greatest time "
          "travel movie ever happened in the history of world cinema. "
          "I love this movie to death, I love, LOVE, love it!")

# ① 정규식 사용 : 알파벳과 공백을 제외한 모든 특수 문자 제거 후 소문자화
review = re.sub("[^a-zA-Z ]", "", review).lower()

# ② 단어를 하나씩 분리하여 단어 사전 조회 후 정수 인덱스 시퀀스로 변환
review_encoding = []
for w in review.split():
    index = word_to_index.get(w, 2)              # 딕셔너리에 없는 단어는 임의로 토큰 인덱스(2) 반환
    if index < 10000:                            # 단어 개수 : 10,000 이하
        review_encoding.append(index)
    else:
        review_encoding.append(word_to_index["<UNK>"])

# ③ 모델에 입력할 수 있도록 2차원 리스트 형태의 100칸짜리 패딩 시퀀스로 고정
test_input = pad_sequences([review_encoding], maxlen=100)

# ④ 최종 긍정/부정 확률 스코어 예측
value = model.predict(test_input)
print(f"예측 확률 스코어: {value[0][0]:.4f}")

if value > 0.5:
    print("최종 판별 결과: [긍정적인 리뷰입니다.]")
else:
    print("최종 판별 결과: [부정적인 리뷰입니다.]")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
예측 확률 스코어: 1.0000
최종 판별 결과: [긍정적인 리뷰입니다.]
